In [9]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [4]:
load_dotenv()
client = OpenAI()

In [5]:
system_prompt = """
You are an expert C++ performance engineer.

Your task is to convert Python code into highly optimized C++ code.

STRICT RULES:
- Output ONLY valid C++ code (no explanations, no markdown, no text outside code)
- The program must compile and run correctly
- Preserve EXACT functionality and output
- Optimize for maximum performance (time complexity and execution speed)
- Use fast I/O where applicable
- Avoid unnecessary abstractions
- Use appropriate data types and memory optimizations
- Include necessary headers

You may include brief inline comments inside the code if needed.
"""
def user_prompt_for(python_code):
    return f"""
Convert the following Python code into optimized C++.

Ensure:
- Identical output
- Maximum performance
- Clean and efficient implementation

Python code:

{python_code}
"""


In [6]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [7]:
messages = [{"role":"system","content":system_prompt},{"role":"system","content":user_prompt_for(pi)}]

In [10]:
response = client.chat.completions.create(
    model="gpt-5",
    messages=messages,
    reasoning_effort="high"
)
result = (response.choices[0].message.content)

In [11]:
print(result)

#include <cstdio>
#include <chrono>
#include <cstdint>

static inline double calculate(uint32_t iterations, int param1, int param2) {
    double result = 1.0;
    const double dp1 = static_cast<double>(param1);
    const double dp2 = static_cast<double>(param2);

    uint32_t i = 1;
    uint32_t lim4 = iterations & ~3u;

    for (; i <= lim4; i += 4) {
        double b, t;

        b = static_cast<double>(i) * dp1;
        t = b - dp2; result -= 1.0 / t;
        t = b + dp2; result += 1.0 / t;

        b = static_cast<double>(i + 1) * dp1;
        t = b - dp2; result -= 1.0 / t;
        t = b + dp2; result += 1.0 / t;

        b = static_cast<double>(i + 2) * dp1;
        t = b - dp2; result -= 1.0 / t;
        t = b + dp2; result += 1.0 / t;

        b = static_cast<double>(i + 3) * dp1;
        t = b - dp2; result -= 1.0 / t;
        t = b + dp2; result += 1.0 / t;
    }

    for (; i <= iterations; ++i) {
        double b = static_cast<double>(i) * dp1;
        double t = b - dp2; r